Session 2 RQ2 recovered full evaluation v2. Genuine all100 B/C outputs; unchanged seeded starter scoring uses matching candidate SETS while model inputs use byte-identical frozen shared ordering. Independent raw-label scoring, paired differences, oracle, fixed summary sample, and one-purchase arithmetic evidence.

In [ ]:
from pathlib import Path
import os,sys,json,csv,math,hashlib,subprocess,time,importlib.util
from collections import defaultdict
OWN=Path.cwd()/"llm4rec_s2_rq2"
INPUT=OWN/"inputs"
ART=OWN/"recovered_v1"
assert (ART/"inference-completed.json").is_file(),"No real full100 outputs: do not run evaluation"
assert not (ART/"evaluation").exists(),"Preserve earlier evaluation; choose a new version"
OUT=ART/"evaluation";OUT.mkdir()
DATA=Path.cwd()/json.loads((OWN/"data-path.json").read_text())["relative"]
protocol=json.loads((OWN/"shared_contexts/protocol.json").read_text())
contexts=[json.loads(l) for l in (OWN/"shared_contexts/contexts.jsonl").read_text().splitlines()]
by_uid={c["user_id"]:c for c in contexts};users=protocol["users"]
assert len(users)==100
truth={r["user_id"]:set(r["truth_ids"]) for r in map(json.loads,(OWN/"evidence/target-reachability.jsonl").read_text().splitlines())}
assert sum(bool(truth[u]) for u in users)==11

raw_truth=defaultdict(set);seen=set()
eligible=set(protocol["eligible_assets"])
with (DATA/"transactions.csv").open(encoding="utf-8-sig",newline="") as f:
    for row in csv.DictReader(f):
        key=tuple(sorted(row.items()))
        if key in seen:continue
        seen.add(key)
        if int(row["transactionID"])<0 or row["transactionType"]!="Buy" or row["ISIN"] not in eligible:continue
        if protocol["cutoff"]<=row["timestamp"][:10]<protocol["end_exclusive"]:raw_truth[row["customerID"]].add(row["ISIN"])
assert all(truth[u]==raw_truth[u] for u in users)

spec=importlib.util.spec_from_file_location("shared",INPUT/"llm_rerank_original.py")
shared=importlib.util.module_from_spec(spec);spec.loader.exec_module(shared)
def score(ids,target):
    dcg=sum(1/math.log2(i+2) for i,a in enumerate(ids[:10]) if a in target)
    ideal=sum(1/math.log2(i+2) for i in range(min(10,len(target))))
    return dcg/ideal if ideal else 0.0
logs={};arm_metrics={};per_user={};predictions={};scoring_times={}
for arm in ["B","C"]:
    logs[arm]=[json.loads(l) for l in (ART/f"full100/{arm}/raw_log.jsonl").read_text().splitlines()]
    responses=[json.loads(l) for l in (ART/f"full100/{arm}/responses.jsonl").read_text().splitlines()]
    assert [r["user_id"] for r in logs[arm]]==[r["user_id"] for r in responses]==users
    for log,response in zip(logs[arm],responses):
        ctx=by_uid[log["user_id"]];shared.assert_no_future(ctx)
        assert log["candidate_ids"]==[a["id"] for a in ctx["candidate_assets"]]
        assert response["protocol_hash"]==protocol["protocol_hash"]
        if log.get("summary_generation"):
            sg=log["summary_generation"]
            assert shared.summary_prompt(ctx,"C",False)==(log["summary_system"],log["summary_prompt"])
            assert sg["response"]==log["summary"]
        if log.get("rank_generation"):
            rg=log["rank_generation"]
            expected=shared.rank_prompt(ctx,shared.history_block(ctx) if arm=="B" else log["summary"])
            assert expected==(log["rank_system"],log["rank_prompt"])
            try:numbers=shared.parse_ranking(rg["response"],50)
            except TypeError:numbers=[]
            assert response["ranked_ids"]==[ctx["candidate_assets"][n-1]["id"] for n in numbers]
        for field in ["summary_generation","rank_generation"]:
            generation=log.get(field)
            if generation:
                request=generation["request"]
                assert generation["key"]==hashlib.sha256(json.dumps(request,sort_keys=True,ensure_ascii=False).encode()).hexdigest()
                assert request["config"]["frozen_contexts_sha256"]==hashlib.sha256((OWN/"shared_contexts/contexts.jsonl").read_bytes()).hexdigest()
                assert generation["input_tokens"]==len(generation["input_ids"])
                assert generation["output_tokens"]==len(generation["output_ids"])
                assert generation["input_tokens"]+request["max_new_tokens"]<=generation["context_limit"]
                assert request["messages"]==[{"role":"system","content":log["summary_system" if field=="summary_generation" else "rank_system"]},{"role":"user","content":log["summary_prompt" if field=="summary_generation" else "rank_prompt"]}]
                assert request["config"]["enable_thinking"] is False and request["config"]["do_sample"] is False
                assert all(banned not in " ".join(m["content"] for m in request["messages"]) for banned in ["riskLevel","investmentCapacity","customerType","lastQuestionnaireDate","assetName",'"truth"','"label"','"future"'])
    cmd=[sys.executable,str(INPUT/"starter.py"),"run","--data-dir",str(DATA),"--output-dir",str(OUT/arm),"--responses",str(ART/f"full100/{arm}/responses.jsonl"),"--stage","ranker","--on-invalid","fallback"]
    t=time.perf_counter();r=subprocess.run(cmd,env=dict(os.environ,PYTHONHASHSEED="0"),capture_output=True,text=True)
    (OUT/f"{arm}-console.log").write_text(r.stdout+r.stderr)
    assert r.returncode==0,r.stdout+r.stderr
    scoring_times[arm]={"command":cmd,"PYTHONHASHSEED":"0","seconds_whole_starter_scoring_run":time.perf_counter()-t}
    scored_contexts=[json.loads(line) for line in (OUT/arm/"contexts.jsonl").read_text().splitlines()]
    assert [c["user_id"] for c in scored_contexts]==users
    for frozen,scored in zip(contexts,scored_contexts):
        assert {a["id"] for a in frozen["candidate_assets"]}=={a["id"] for a in scored["candidate_assets"]}
        assert dict(frozen,candidate_assets=sorted(frozen["candidate_assets"],key=lambda x:x["id"]))==dict(scored,candidate_assets=sorted(scored["candidate_assets"],key=lambda x:x["id"]))
    assert json.loads((OUT/arm/"protocol.json").read_text())==protocol
    m=json.loads((OUT/arm/"metrics.json").read_text());arm_metrics[arm]=m
    baseline=json.loads((OWN/"baseline_seed0/metrics.json").read_text())
    for method,values in baseline["methods"].items():assert m["methods"][method]==values
    failures={f["user_id"] for f in m["custom_failures"]}
    assert failures=={l["user_id"] for l in logs[arm] if not l["valid"]}
    predictions[arm]=defaultdict(list)
    with (OUT/arm/"predictions.csv").open() as f:
        for row in csv.DictReader(f):
            if row["method"]=="custom":predictions[arm][row["user_id"]].append(row["ISIN"])
    per_user[arm]={}
    with (OUT/arm/"per_user.csv").open() as f:
        for row in csv.DictReader(f):
            if row["method"]=="custom":per_user[arm][row["user_id"]]=float(row["ndcg_at_10"])
    for u in users:
        ids=predictions[arm][u]
        assert len(ids)==len(set(ids))==10 and set(ids)<={a["id"] for a in by_uid[u]["candidate_assets"]}
        value=score(ids,truth[u]);assert abs(value-per_user[arm][u])<1e-14
    values=list(per_user[arm].values());nonempty=[per_user[arm][u] for u in users if truth[u]]
    assert abs(sum(values)/100-m["methods"]["custom"]["ndcg_at_10"])<1e-14
    assert abs(sum(nonempty)/11-m["methods"]["custom"]["ndcg_at_10_nonempty"])<1e-14
assert [r["candidate_ids"] for r in logs["B"]]==[r["candidate_ids"] for r in logs["C"]]
paired=[{"user_id":u,"truth_count":len(truth[u]),"B":per_user["B"][u],"C":per_user["C"][u],"C_minus_B":per_user["C"][u]-per_user["B"][u]} for u in users]
with (OUT/"paired-user-differences.csv").open("x",newline="") as f:
    w=csv.DictWriter(f,fieldnames=list(paired[0]));w.writeheader();w.writerows(paired)
eligible_examples=[u for u in users if len(truth[u])==1 and predictions["B"][u]!=predictions["C"][u]]
assert eligible_examples,"No one-future-purchase user with different B/C rankings; inspect actual outputs"
example_uid=max(eligible_examples,key=lambda u:abs(per_user["C"][u]-per_user["B"][u]))
example={"user_id":example_uid,"student_personally_verified":False,"formula":"For a single future purchase, IDCG=1; nDCG@10=1/log2(rank+1) if target is within top10, otherwise zero.","cutoff":protocol["cutoff"],"end_exclusive":protocol["end_exclusive"],"truth_ids":sorted(truth[example_uid]),"ideal_dcg_at_10":sum(1/math.log2(i+2) for i in range(min(10,len(truth[example_uid])))),"arms":{}}
for arm in ["B","C"]:
    example["arms"][arm]={"target_rank_in_evaluated_top10":next((i+1 for i,a in enumerate(predictions[arm][example_uid]) if a in truth[example_uid]),None),"raw_model_ranked_ids":next(l for l in logs[arm] if l["user_id"]==example_uid)["ranked_ids"],"starter_fallback_used":any(f["user_id"]==example_uid for f in arm_metrics[arm]["custom_failures"]),"ranking":predictions[arm][example_uid],"contributions":[{"rank":i+1,"id":a,"gain":1/math.log2(i+2) if a in truth[example_uid] else 0.0} for i,a in enumerate(predictions[arm][example_uid])],"ndcg_at_10":per_user[arm][example_uid],"raw_log":next(l for l in logs[arm] if l["user_id"]==example_uid)}
(OUT/"student-check-example.json").write_text(json.dumps(example,indent=2)+"\n")
summary={"status":"full100_executed_and_verified","protocol_hash":protocol["protocol_hash"],"baseline":baseline["methods"],"B":arm_metrics["B"]["methods"]["custom"],"C":arm_metrics["C"]["methods"]["custom"],"invalid_outputs":{a:len(arm_metrics[a]["custom_failures"]) for a in ["B","C"]},"paired":{"all_users_mean_C_minus_B":sum(r["C_minus_B"] for r in paired)/100,"nonempty_mean_C_minus_B":sum(r["C_minus_B"] for r in paired)/11,"nonempty_C_wins":sum(r["C_minus_B"]>0 for r in paired if r["truth_count"]),"nonempty_C_losses":sum(r["C_minus_B"]<0 for r in paired if r["truth_count"]),"nonempty_ties":sum(r["C_minus_B"]==0 for r in paired if r["truth_count"]),"inference":"descriptive paired comparison; only 11 nonempty users, no significance claim"},"generation_timings":{a:json.loads((ART/f"full100/{a}/run_meta.json").read_text()) for a in ["B","C"]},"cold_load_and_hardware":json.loads((ART/"model-hardware-metadata.json").read_text()),"scoring_timings":scoring_times,"oracle":json.loads((OWN/"evidence/candidate-oracle.json").read_text())}
summary["recovery"]={"first_turn_status":"network_blocked_no_BC_inference","this_run":"genuine offline recovered B/C inference","offline_install":json.loads((OWN/"offline_environment.json").read_text()),"shared_input_provenance":json.loads((OWN/"shared_contexts/provenance.json").read_text()),"shared_context_sha256":hashlib.sha256((OWN/"shared_contexts/contexts.jsonl").read_bytes()).hexdigest(),"shared_context_parity":json.loads((OWN/"shared-context-parity.json").read_text()),"no_teammate_model_results_used":True}
summary["baseline_timing"]=json.loads((OWN/"seeded-baseline-manifest.json").read_text())["runs"][0]
summary["smoke_timings"]={a:json.loads((ART/f"smoke/{a}/run_meta.json").read_text()) for a in ["B","C"]}
summary["counts"]={"all_users":100,"nonempty_truth":11,"empty_truth":89}
summary["student_check_user"]=example_uid
summary["context_token_stats"]={}
for arm in ["B","C"]:
    summary["context_token_stats"][arm]={}
    for purpose in ["summary_generation","rank_generation"]:
        gens=[l[purpose] for l in logs[arm] if l.get(purpose)]
        if gens:
            summary["context_token_stats"][arm][purpose]={"calls":len(gens),"input_min":min(g["input_tokens"] for g in gens),"input_max":max(g["input_tokens"] for g in gens),"input_total":sum(g["input_tokens"] for g in gens),"output_min":min(g["output_tokens"] for g in gens),"output_max":max(g["output_tokens"] for g in gens),"output_total":sum(g["output_tokens"] for g in gens),"new_token_cap_hits":sum(g["hit_new_token_cap"] for g in gens)}
# Confirm the saved oracle again with the frozen inputs and freshly rederived labels.
reachable=[]
for u in users:
    candidate_ids=[a["id"] for a in by_uid[u]["candidate_assets"]]
    hits=truth[u]&set(candidate_ids)
    oracle=sorted(hits)+[i for i in candidate_ids if i not in hits]
    reachable.append({"user_id":u,"truth_count":len(truth[u]),"reachable":len(hits),"oracle_ndcg_at_10":score(oracle[:10],truth[u])})
assert abs(sum(r["oracle_ndcg_at_10"] for r in reachable)/100-summary["oracle"]["top50_oracle_ndcg_at_10_all"])<1e-14
with (OUT/"candidate-oracle-per-user.jsonl").open("x") as f:
    for row in reachable:f.write(json.dumps(row)+"\n")
# A fixed, predeclared sample. This is summary inspection only; prompts are never changed from this audit.
audit_plan=json.loads((OWN/"summary-audit-plan.json").read_text())
audit=[]
for index in audit_plan["protocol_user_indices_zero_based"]:
    uid=users[index];log=logs["C"][index];ctx=by_uid[uid]
    assert log["user_id"]==uid
    shown=ctx["history"][:shared.HISTORY_CAP]
    market_counts=defaultdict(int);category_counts=defaultdict(int)
    for asset in shown:
        for attribute in asset["attributes"]:
            field,value=attribute.split("=",1)
            if field=="marketID":market_counts[value]+=asset["buy_count"]
            if field=="assetCategory":category_counts[value]+=asset["buy_count"]
    audit.append({"user_id":uid,"protocol_index":index,"summary":log["summary"],"generation":log["summary_generation"],"history_as_shown":shown,"shown_buy_count":sum(a["buy_count"] for a in shown),"history_assets_total":len(ctx["history"]),"market_buy_counts_shown":dict(market_counts),"category_buy_counts_shown":dict(category_counts),"prompts_changed_after_audit":False})
with (OUT/"fixed-summary-audit-inputs.jsonl").open("x") as f:
    for row in audit:f.write(json.dumps(row,ensure_ascii=False)+"\n")
(OUT/"results-summary.json").write_text(json.dumps(summary,indent=2)+"\n")
(OUT/"verification.json").write_text(json.dumps({"status":"passed","all100_in_both_arms":True,"actual_prompts_equal_shared_templates":True,"no_supplied_future_name_profile_fields":True,"generation_cache_prompt_hashes":True,"context_no_silent_truncation":True,"same_ordered_frozen_inputs_B_C":True,"candidate_sets_equal_in_unchanged_starter_scoring":True,"invalid_users_retained_with_starter_fallback":True,"output_uniqueness_eligibility":True,"independent_saved_prediction_ndcg_recomputation":True,"paired_differences_saved":True,"personal_verification_claimed":False},indent=2)+"\n")
print(json.dumps({a:summary[a] for a in ["B","C","paired"]},indent=2))
